# Apple on-device LLM — 2K Colab render

One **16:9 YouTube master**, rendered at **2560×1440 (2K)**, 30 fps, H.264 + AAC.

This notebook is written for a **2-core Colab CPU runtime**:

- Google Drive is mounted **first**, so Colab asks for access at the beginning.
- Clones the public repo `lroe/apple-dracula`.
- Uses `CONCURRENCY=2`, `veryfast` x264 and JPEG frame capture to stretch the two cores.
- Streams **frame count, percent, fps, elapsed time and ETA** every ~3 seconds.
- Automatically copies the finished MP4, render log and manifest to `MyDrive/apple-render/`.
- Leave this tab open. A free Colab runtime can still disconnect after several hours.


## 1. Mount Google Drive first

Authorize the Drive prompt now. The finished render will go to `MyDrive/apple-render/`.


In [ ]:
from google.colab import drive
from pathlib import Path
import shutil

drive.mount('/content/drive', force_remount=False)

DRIVE_DIR = Path('/content/drive/MyDrive/apple-render')
DRIVE_DIR.mkdir(parents=True, exist_ok=True)
print(f'Drive ready: {DRIVE_DIR}')
print(f'Drive free space: {shutil.disk_usage("/content/drive").free / (1024**3):.1f} GB')


## 2. Install Node, ffmpeg and headless-Chrome dependencies


In [ ]:
%%bash
set -euo pipefail

apt-get -qq update
apt-get -qq install -y ffmpeg git curl ca-certificates

# Node 20 for Remotion 4
curl -fsSL https://deb.nodesource.com/setup_20.x | bash -
apt-get -qq install -y nodejs

# Headless Chrome runtime libraries
apt-get -qq install -y \
  libnss3 libatk1.0-0 libatk-bridge2.0-0 libcups2 libdrm2 \
  libxkbcommon0 libxcomposite1 libxdamage1 libxfixes3 libxrandr2 \
  libgbm1 libasound2 libpango-1.0-0 libcairo2 || true

node -v
npm -v
ffmpeg -version | head -1
nproc


## 3. Clone the public repo


In [ ]:
import os, shutil, subprocess
from pathlib import Path

REPO_URL = 'https://github.com/lroe/apple-dracula.git'
REPO_DIR = Path('/content/apple-dracula')
VIDEO_DIR = REPO_DIR / 'video'

if (REPO_DIR / '.git').exists():
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only'], check=True)
else:
    if REPO_DIR.exists():
        shutil.rmtree(REPO_DIR)
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(REPO_DIR)], check=True)

COMMIT = subprocess.check_output(
    ['git', '-C', str(REPO_DIR), 'rev-parse', '--short', 'HEAD'],
    text=True,
).strip()

print('Repo   :', REPO_URL)
print('Commit :', COMMIT)
print('Video  :', VIDEO_DIR)


## 4. Install npm dependencies and Chrome


In [ ]:
%cd /content/apple-dracula/video
!npm ci --no-audit --no-fund
!npx remotion browser ensure


## 5. Live-output runner + Drive uploader

Everything below streams output immediately, so Colab shows render progress while
the cell runs. The uploader is also defined here and is called automatically after
the full render finishes.


In [ ]:
import json, os, shutil, subprocess, time
from pathlib import Path

VIDEO_DIR = Path('/content/apple-dracula/video')
LOG_DIR = Path('/content/render-logs')
LOG_DIR.mkdir(parents=True, exist_ok=True)


def run_stream(args, log_name):
    """Run a command and print every stdout line immediately."""
    env = os.environ.copy()
    env['CONCURRENCY'] = '2'
    env['PROGRESS_INTERVAL_MS'] = '3000'
    env['PYTHONUNBUFFERED'] = '1'

    log_path = LOG_DIR / log_name
    print(f'Running: {" ".join(args)}', flush=True)
    print(f'Log    : {log_path}', flush=True)
    started = time.time()

    with open(log_path, 'w', buffering=1) as log:
        proc = subprocess.Popen(
            args,
            cwd=str(VIDEO_DIR),
            env=env,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
        )
        assert proc.stdout is not None
        for line in proc.stdout:
            log.write(line)
            print(line, end='', flush=True)
        return_code = proc.wait()

    elapsed = time.time() - started
    print(f'\nExit code: {return_code} | wall time: {elapsed / 60:.1f} min', flush=True)
    if return_code != 0:
        raise RuntimeError(f'Command failed ({return_code}). See {log_path}')
    return log_path


def upload_render_to_drive():
    """Copy the finished render, log and manifest to Google Drive."""
    src = VIDEO_DIR / 'out/apple-explainer-2k.mp4'
    if not src.exists():
        raise FileNotFoundError(f'Render not found: {src}')

    drive_dir = Path('/content/drive/MyDrive/apple-render')
    drive_dir.mkdir(parents=True, exist_ok=True)
    dst = drive_dir / src.name
    part = dst.with_suffix(dst.suffix + '.part')

    size_mb = src.stat().st_size / (1024 * 1024)
    print(f'Local render: {src} ({size_mb:.1f} MB)', flush=True)

    probe = subprocess.check_output(
        [
            'ffprobe', '-v', 'error',
            '-select_streams', 'v:0',
            '-show_entries', 'stream=width,height,r_frame_rate,codec_name,bit_rate',
            '-of', 'default=nw=1',
            str(src),
        ],
        text=True,
    )
    print(probe, flush=True)

    def copy_with_progress(source, target):
        total = source.stat().st_size
        copied = 0
        last_report = 0.0
        with open(source, 'rb') as reader, open(target, 'wb') as writer:
            while True:
                chunk = reader.read(16 * 1024 * 1024)
                if not chunk:
                    break
                writer.write(chunk)
                copied += len(chunk)
                now = time.time()
                if now - last_report >= 10 or copied == total:
                    pct = copied / total * 100 if total else 100
                    print(
                        f'Upload: {pct:5.1f}% | {copied / (1024**3):.2f} GB / {total / (1024**3):.2f} GB',
                        flush=True,
                    )
                    last_report = now
            writer.flush()

    uploaded = False
    for attempt in range(1, 4):
        try:
            print(f'Uploading to Drive (attempt {attempt}/3)...', flush=True)
            if part.exists():
                part.unlink()
            copy_with_progress(src, part)
            try:
                os.replace(part, dst)
            except OSError:
                shutil.move(str(part), str(dst))
            uploaded = True
            break
        except Exception as exc:
            print(f'Upload attempt {attempt} failed: {exc}', flush=True)
            time.sleep(15)

    if not uploaded:
        raise RuntimeError(f'Could not copy the render to Drive: {dst}')

    shutil.copyfile(LOG_DIR / 'render-2k.log', drive_dir / 'render-2k.log')

    manifest = {
        'repo': REPO_URL,
        'commit': COMMIT,
        'rendered_at': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()),
        'resolution': '2560x1440',
        'fps': 30,
        'codec': 'h264',
        'audio': 'aac 320kbps',
        'file': str(dst),
        'size_bytes': dst.stat().st_size,
    }
    (drive_dir / 'render-2k-manifest.json').write_text(
        json.dumps(manifest, indent=2), encoding='utf-8'
    )
    return dst


print('Runner ready. Uploader ready.')


## 6. Quick 2K test — 12 frames

Validates 2560×1440, audio, browser, and the live progress stream before the long render.


In [ ]:
test_log = run_stream(['node', 'render-2k.mjs', '--test'], 'render-2k-test.log')
print('Test log:', test_log)


## 7. Full 2K render + automatic Drive upload

This is the long step. Every ~3 seconds the output prints:

`[elapsed] percent | frames done/total | encoded | fps | ETA`

When it finishes, the same cell automatically copies the MP4, log and manifest to
`MyDrive/apple-render/`. Keep the tab open and the machine awake.


In [ ]:
try:
    full_log = run_stream(['node', 'render-2k.mjs'], 'render-2k.log')
    print('Full log:', full_log)
    uploaded_path = upload_render_to_drive()
    print('==============================================')
    print('DONE — render uploaded to Google Drive')
    print(uploaded_path)
    print('==============================================')
except Exception:
    # If the render or upload dies, still put the log in Drive so it can be diagnosed.
    try:
        drive_dir = Path('/content/drive/MyDrive/apple-render')
        if drive_dir.exists():
            shutil.copyfile(LOG_DIR / 'render-2k.log', drive_dir / 'render-2k.log')
            print('Failure log copied to Drive.', flush=True)
    except Exception as log_exc:
        print(f'Could not copy failure log: {log_exc}', flush=True)
    raise


## 8. Manual re-upload (optional)

If the render finished but the Drive copy was interrupted, run the uploader again
without re-rendering anything.


In [ ]:
# upload_render_to_drive()


## YouTube upload

Upload `apple-explainer-2k.mp4` as-is. YouTube will keep the 2560×1440 master and transcode to VP9/AV1 for playback. Do not re-encode it before uploading.
